In [ ]:
import os
from IPython.display import clear_output

notebook_dir = "/home/balabaevvl/courses/nlp/nlp_course/week10_agents/seminar"  # notebook's dir
os.chdir(notebook_dir)

print("Current dir:", os.getcwd())


In [ ]:
import os

gpu0 = "GPU-e83bd31b-fcb9-b8de-f617-2d717619413b"     # 0
gpu1 = "GPU-5a9b7750-9f85-49a5-3aae-fe07b1b7661d"     # 1
gpu2 = "GPU-fe2d8dfd-06f2-a5c4-a7fd-4a5f23947005"     # 2
gpu3 = "GPU-0c320096-21ee-4060-8731-826ca2febfab"     # 3
gpu4 = "GPU-baef952c-6609-aace-3b78-e4e07788d5de"     # 4
gpu5 = "GPU-3979d65b-c238-4e9c-0c1c-1aa3f05c56a1"     # 5
gpu6 = "GPU-6c76a2c5-5375-aa06-11d4-0fddfac30e91"     # 6
os.environ["CUDA_VISIBLE_DEVICES"] = f"{gpu0}"


# import torch  # comment out, if you use `bitsandbytes`
# device = torch.device('cuda:0')

In [ ]:
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"       # !pip install -U hf_transfer
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"    # problems with progress bar

# import shutil
# from huggingface_hub.constants import HF_HUB_CACHE
# cache = os.path.expanduser(HF_HUB_CACHE)
# for d in os.listdir(cache):
#     if d.startswith("models--unsloth--Llama-3.2-3B"):
#         shutil.rmtree(os.path.join(cache, d), ignore_errors=True)
#         print(f"Deleted {d}")


# --- DELETE ALL CACHE ---
# import os, shutil
# from huggingface_hub import scan_cache_dir
# from huggingface_hub.constants import HF_HUB_CACHE

# cache_info = scan_cache_dir()   # Scans everything in ~/.cache/huggingface/hub

# for repo in cache_info.repos:
#     print(f"Deleting cache for: {repo.repo_id}")
#     shutil.rmtree(repo.repo_path, ignore_errors=True)


In [1]:
!pip install "vllm>=0.8.5" mcp ddgs smolagents markdownify

from IPython.display import clear_output
clear_output()

In [2]:
import typing as tp
import subprocess

# Model setup

**Visit https://openrouter.ai/ to create an account, generate api key and search for models**

In [3]:
PROVIDER = 'OpenRouter' # OpenRouter or vLLM
# PROVIDER = 'vLLM' # OpenRouter or vLLM

In [4]:
from getpass import getpass

if PROVIDER == 'vLLM':
    KEY = "EMPTY"
    HOST = "127.0.0.1"
    PORT = "8999"
    MODEL = "Qwen/Qwen3-4B-Instruct-2507"
    URL = f"http://{HOST}:{PORT}/v1"
else:
    MODEL = "x-ai/grok-4.1-fast:free"
    KEY = getpass("Your openrouter api key: ")
    URL = f"https://openrouter.ai/api/v1"


Your openrouter api key: ··········


In [5]:
if PROVIDER == 'vLLM':
    vllm_server = subprocess.Popen([
        "python",
        "-m", "vllm.entrypoints.openai.api_server",
        "--model", MODEL,
        "--max_model_len", "16384",
        "--host", HOST,
        "--port", PORT,
        "--gpu_memory_utilization", "0.6",
        "--max_num_seqs", "16",
    ])
    #  python -m vllm.entrypoints.openai.api_server --model Qwen/Qwen3-4B-Instruct-2507 --max_model_len 16384 --host 127.0.0.1 --port 8999 --gpu_memory_utilization 0.6 --max_num_seqs 16 --temperature 0.9

# vllm_server.terminate() — остановить


In [6]:
from openai import OpenAI

client = OpenAI(
  base_url=URL,
  api_key=KEY,
)


response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Hi"}],
)

print(response.choices[0].message)


ChatCompletionMessage(content="Hi! What's up?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None, reasoning_details=[{'id': 'rs_66eb5705-1a31-4fbc-3f54-96308939a096', 'format': 'xai-responses-v1', 'index': 0, 'type': 'reasoning.encrypted', 'data': 'rXzWEF4CXOmXzb0n+fTxbsSt4CMsEjrcrSc7+xc/D4GTCtADfaXLU05tpxu9FQMfzNCpMzxzIn1acYHP0MVqL/zsVzuqx7RGBRSc7U5Jv2/Nm9WZx/j/5ecjchpFH+MyUOurqobucZFoDzJG+UVCJUR2fPFhqzqhx+QyPwF0FFyrWjvBiFn/J/xvctE6nmR4iCY/+snXPl1FqgkTPpPdoZeTnvmiM9RdC+sbiAR0U5k7gfrnns35PyWi9bJAYIIHzmQoqQRzzfW+tXE6mU6iS9QZtM8O34h78GLJC0r3np2uaiHKtuSLLBcRfsX+01o5OjyGKSGGDPXxL4j3LrBxVMbYjUetZaEkpskHIp3VKEmtSCAyhRy894gDtJddAruuEXsPkU68ZwrkjYWM8vvV9xpRD9X4VI3s15IZyW0NCoi4MUPnvc92PSiDB2u7R6SpPKCi9GFW9DwxSJRkXIh+TM1aQl2vkUgMlDQaixq9/7Zlo4jXqWJFcT0bGEOQfu7Hy3w9AgQuFfWNyOp3WUc+lO8ugQI8E0dbkeOmQOuLopFyIZwJMz1/ezf6bqfEkbdRfuVMMhATsal8RAdS3IgWq/IotcUFapA5KuhyPy1J+TnbXw1DUlvuk4udmfXuq690oa62YUmVapCRbfF0/9jI16WoDgaBljMs+fbfhWrmUeGLtC

# How to write tools? How to use MCP?

## Tools as functions

In [38]:
from ddgs import DDGS

def duckduckgo_search(query: str, max_results: int = 5) -> list[dict[str, str]]:
    """

    """
    with DDGS() as ddgs:
        results = ddgs.text(query, max_results=max_results, )

    return list(results)


In [8]:
duckduckgo_search('yandex data school')

[{'title': 'School of data analysis',
  'href': 'https://dataschool.yandex.com/',
  'body': "The two-year Yandex program was created in 2007 and has become Russia's leading data analysis program. Courses from the Yandex School of Data Analysis serve as the foundation for Master's programs at major universities, such as the Higher School of Economics and the Moscow Institute of Physics and Technology."},
 {'title': 'Yandex School of Data Analysis - GitHub',
  'href': 'https://github.com/yandexdataschool',
  'body': 'Yandex School of Data Analysis has 114 repositories available. Follow their code on GitHub.'},
 {'title': 'Yandex school of Data Analysis. About a school with Machine Learning ...',
  'href': 'https://awant.medium.com/yandex-school-of-data-analysis-5d30d018912e',
  'body': 'In the 2000s, when Yandex flourished, it was hard to find excellent specialists in the labor market and the need to solve problems related to data processing (text, images, music, voice) was constantly gr

In [14]:
import re
import requests
from markdownify import markdownify
from requests.exceptions import RequestException


def get_webpage_content(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()

        # Convert the HTML content to Markdown
        markdown_content = markdownify(response.text).strip()
        # Remove multiple line breaks
        markdown_content = re.sub(r"\n{3,}", "\n\n", markdown_content)

        return markdown_content

    except RequestException as e:
        return f"Error fetching the webpage: {str(e)}"
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

In [21]:
# Implement function to get top-5 results with full text
def websearch_full_text(query, top_k=1) -> list[dict[str, str]]:
    searches = duckduckgo_search(query, max_results=top_k)

    searches = [{**search, "body": get_webpage_content(search["href"])} for search in searches]

    return searches


In [54]:
websearch_full_text("trump blows bubba", 1)

[{'title': 'Donald Trump ‘ blowing Bubba ’ message in Epstein... - Newsweek',
  'href': 'https://www.newsweek.com/donald-trump-blowing-bubba-message-epstein-emails-under-scrutiny-11046836',
  'body': 'Donald Trump ‘blowing Bubba’ message in Epstein emails under scrutiny - Newsweek\n\nLive Updates\n\n[Thanksgiving Winter Storm Live Tracker: What is \'lake effect\' snow? Warnings for New York, Pennsylvania, and Ohio](/thanksgiving-winter-storm-live-tracker-updates-as-heavy-snow-to-hit-11111003)\n\n* [Nation](/us)\n  + [News](/news)\n  + [Politics](/politics)\n  + [Tech](/technology)\n  + [Fact Check](/fact-check)\n  + [Personal Finance](/personal-finance)\n  + [Automotive](/autos)\n  + [Sports](/sports)\n  + [Better Workplaces](/better-workplaces)\n* [World](/world)\n  + [Russia-Ukraine](/topic/russia)\n  + [Middle East](/topic/middle-east)\n  + [China And Asia](/topic/china)\n  + [Better Planet](/better-planet)\n  + [All World News](/world)\n* [Lifestyle](/life)\n  + [Family & Parenting

## Run MCP Server

## Connect MCP client

In [24]:
import asyncio
from typing import Optional
from contextlib import AsyncExitStack

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

In [39]:
class MCPClient:
    def __init__(self, mcps: list[str]):
        self.session: Optional[ClientSession] = None
        self.exit_stack = AsyncExitStack()
        self.mcps = mcps
        self.tools = []

    async def connect_to_server(self, server_script_path: str):
        server_params = StdioServerParameters(
            command="python",
            args=[server_script_path],
            env=None
        )

        self.stdio, self.write = await self.exit_stack.enter_async_context(stdio_client(server_params, errlog=None))
        self.session = await self.exit_stack.enter_async_context(ClientSession(self.stdio, self.write))
        await self.session.initialize()

        response = await self.session.list_tools()
        self.tools = response.tools

    def list_tools(self):
        print("\nConnected to server with tools:", [tool.name for tool in self.tools])
        return self.tools

    async def call_tool(self, tool_name, args):
        result = await self.session.call_tool(tool_name, args)
        return result



In [40]:
client = MCPClient([])
await client.connect_to_server('ysda_tools.py')
tools = client.list_tools()


Connected to server with tools: ['add', 'subtract', 'multiply', 'divide', 'vector_add', 'vector_subtract', 'vector_dot', 'vector_elementwise_multiply', 'matrix_add', 'matrix_subtract', 'matrix_multiply', 'matrix_transpose']


In [55]:
tools[0]

Tool(name='add', title=None, description='', inputSchema={'properties': {'a': {'title': 'A', 'type': 'number'}, 'b': {'title': 'B', 'type': 'number'}}, 'required': ['a', 'b'], 'title': 'addArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'addOutput', 'type': 'object'}, icons=None, annotations=None, meta=None)

# Tool calls example

In [56]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_youtube_captions",
            "description": "Fetch YouTube captions for a given video ID",
            "parameters": {
                "type": "object",
                "properties": {
                    "video_id": {"type": "string"},
                    "lang": {"type": "string", "default": "en"}
                },
                "required": ["video_id"]
            }
        }
    }
]


In [57]:
import inspect
import typing

PYTHON_TO_JSON = {
    str: "string",
    int: "integer",
    float: "number",
    bool: "boolean",
    list: "array",
    dict: "object",
}

def create_tool_description(func):
    sig = inspect.signature(func)
    doc = inspect.getdoc(func) or ""
    params_schema = {"type": "object", "properties": {}, "required": []}

    for name, param in sig.parameters.items():
        entry = {"type": PYTHON_TO_JSON.get(param.annotation, "string")}

        if param.default is inspect.Parameter.empty:
            params_schema["required"].append(name)
        else:
            entry["default"] = param.default

        params_schema["properties"][name] = entry

    return {
        "type": "function",
        "function": {
            "name": func.__name__,
            "description": doc,
            "parameters": params_schema,
        },
    }


In [58]:
create_tool_description(duckduckgo_search)

{'type': 'function',
 'function': {'name': 'duckduckgo_search',
  'description': '    \n    ',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string'},
    'max_results': {'type': 'integer', 'default': 5}},
   'required': ['query']}}}

# ReAct Agent from scratch

In [ ]:
instruction = """Solve a question answering task with interleaving Thought, Action, Observation steps. Thought can reason about the current situation, and Action can be three types:
(1) Search[query], which searches the web for the query and returns url, title and small snippet for 5 relevant pages.
(2) Visit web-page[link], which returns content of the page provided.
(3) Finish[answer], which returns the answer and finishes the task.
"""


final_prompt = "YOUR PROMPT WITH FEW SHOTS"

In [ ]:
def llm(prompt, stop=["\n"]):
    response = client.chat.completions.create(
      model=MODEL,
      prompt=prompt,
      temperature=0,
      max_tokens=100,
      top_p=1,
      frequency_penalty=0.0,
      presence_penalty=0.0,
      stop=stop
    )
    return response["choices"][0]["text"]

In [ ]:
def execute_action(action: str) -> str:
    pass


def call_react(question, prompt=final_prompt, to_print=True):
    prompt += question + "\n"
    n_calls, n_badcalls = 0, 0
    for i in range(1, 8):
        n_calls += 1
        # Generate thought
        thought_action = <YOUR CODE HERE>
        try:
            # Extract generated action
            thought, action = thought_action.strip().split(f"\nAction {i}: ")
        except Exception as e:
            # Error handling
        # Execute action
        ... = execute_action()
        # Create observation
        obs = obs.replace('\\n', '')
        step_str = f"Thought {i}: {thought}\nAction {i}: {action}\nObservation {i}: {obs}\n"
        # Add observation to history
    return r, info

# Multi-agent systems with smolagents

In [60]:
from smolagents import tool


@tool
def visit_webpage(link: str) -> str:
  """
  Visit web-page[link], which returns content of the page provided.
  Args:
    link: link to the webpage
  Returns:
    content of the webpage
  """
  return get_webpage_content(link)

In [61]:
from smolagents import (
    CodeAgent,
    OpenAIModel,
    ToolCallingAgent,
    WebSearchTool,
)


model = OpenAIModel(
    model_id=MODEL,
    api_base=URL,
    api_key=KEY,
)


web_agent = CodeAgent(
    tools=[WebSearchTool(), visit_webpage],
    model=model,
    max_steps=5,
    name="web_search_agent",
    description="Runs web searches for you.",
)

In [62]:
manager_agent = CodeAgent(
    tools=[],
    model=model,
    managed_agents=[web_agent],
)

In [63]:
answer = manager_agent.run(
    "How does ReAct agent works? What metrics were reported by authors?"
)


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ How does ReAct agent works? What metrics were reported by authors?                                              │
│                                                                                                                 │
╰─ OpenAIModel - x-ai/grok-4.1-fast:free ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = web_search_agent(                                                                                       
      task="""Locate the original academic paper on 'ReAct: Synergizing Reasoning and Acting in Language Models'   
  by Shunyu Yao et al. (likely from 2022, Google/Princeton). Provide:                                              
  1. A clear step-by-step explanation of how the ReAct agent works, including the prompt structure,                
  reasoning-acting loop, verbalizer, and integration with external tools/world models.                             
  2. Key metrics reported by the authors, such as exact match/EM/F1 scores on benchmarks like HotpotQA, FeverQA,   
  ALFWorld, WebShop, CRAIG, and comparisons to baselines like Chain-of-Thought (CoT). Include PaLM-540B and other  
  model sizes if mentioned. Quote tables or numbers directly where possible.                                       
  Focus on arXiv, official sites, or NeurIPS/ICLR proceedings. Be precise and comprehensive.""",                   
      additional_args={}                                                                                           
  )                                                                                                                
  print(result)                                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── New run - web_search_agent ───────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'web_search_agent'.                                                                │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ Locate the original academic paper on 'ReAct: Synergizing Reasoning and Acting in Language Models' by Shunyu    │
│ Yao et al. (likely from 2022, Google/Princeton). Provide:                                                       │
│ 1. A clear step-by-step explanation of how the ReAct agent works, including the prompt structure,               │
│ reasoning-acting loop, verbalizer, and integration with external tools/world models.                            │
│ 2. Key metrics reported by the authors, such as exact match/EM/F1 scores on benchmarks like HotpotQA, FeverQA,  │
│ ALFWorld, WebShop, CRAIG, and comparisons to baselines like Chain-of-Thought (CoT). Include PaLM-540B and other │
│ model sizes if mentioned. Quote tables or numbers directly where possible.                                      │
│ Focus on arXiv, official sites, or NeurIPS/ICLR proceedings. Be precise and comprehensive.                      │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│                                                                                                                 │
╰─ OpenAIModel - x-ai/grok-4.1-fast:free ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

...

[Step 1: Duration 66.79 seconds| Input tokens: 2,249 | Output tokens: 646]

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 2 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("""                                                                                                 
  ### How the ReAct Agent Works                                                                                    
                                                                                                                   
  ReAct (Reasoning + Acting in Language Models) is a prompting framework that synergizes verbal reasoning          
  (thoughts) and task-specific actions in large language models (LLMs like PaLM-540B or GPT-3). It interleaves     
  **Thought** (free-form reasoning traces) and **Action** (e.g., `search[entity]`) with **Observation** from       
  external tools/world models, enabling dynamic interaction to solve complex tasks like multi-hop QA, fact         
  verification, and decision-making in simulated environments.                                                     
                                                                                                                   
  #### Key Components:                                                                                             
  1. **Prompt Structure**: Few-shot in-context learning (1-6 examples per task). Each example shows a full         
  trajectory:                                                                                                      
     ```                                                                                                           
     Question: ...?                                                                                                
     Thought: I need to find X by searching Y first.                                                               
     Action: search[Y]                                                                                             
     Observation: [Environment response, e.g., Wikipedia snippet]                                                  
     Thought: From the observation, Z is related...                                                                
     Action: lookup[key_term]                                                                                      
     Observation: ...                                                                                              
     Thought: The answer is A.                                                                                     
     Finish[A]                                                                                                     
     ```                                                                                                           
     Thoughts are natural language for planning, fact extraction, progress tracking, and error handling. Actions   
  are task-specific (via verbalizer).                                                                              
                                                                                                                   
  2. **Reasoning-Acting Loop**:                                                                                    
     - **Generation**: LLM autoregressively generates until a [Thought] or [Action] tag.                           
     - **Dense Mode** (e.g., HotpotQA/FEVER): Strict alternation Thought → Action → Observation (fixed steps).     
     - **Sparse Mode** (e.g., ALFWorld/WebShop): Model decides when to think or act asynchronously.                
     - **Synergy**:                                                                                                
       - Reason → Act: Thoughts guide actions (planning, reformulating queries).                                   
       - Act → Reason: Observations ground reasoning (reduce hallucinations).                                      
     - Loop until `Finish[answer]` or max steps.         

Final answer: 
### How the ReAct Agent Works

ReAct (Reasoning + Acting in Language Models) is a prompting framework that synergizes verbal reasoning (thoughts) 
and task-specific actions in large language models (LLMs like PaLM-540B or GPT-3). It interleaves **Thought** 
(free-form reasoning traces) and **Action** (e.g., `search[entity]`) with **Observation** from external tools/world
models, enabling dynamic interaction to solve complex tasks like multi-hop QA, fact verification, and 
decision-making in simulated environments.

#### Key Components:
1. **Prompt Structure**: Few-shot in-context learning (1-6 examples per task). Each example shows a full 
trajectory:
   ```
   Question: ...?
   Thought: I need to find X by searching Y first.
   Action: search[Y]
   Observation: [Environment response, e.g., Wikipedia snippet]
   Thought: From the observation, Z is related...
   Action: lookup[key_term]
   Observation: ...
   Thought: The answer is A.
   Finish[A]
   ```
   Thoughts are natural language for planning, fact extraction, progress tracking, and error handling. Actions are 
task-specific (via verbalizer).

2. **Reasoning-Acting Loop**:
   - **Generation**: LLM autoregressively generates until a [Thought] or [Action] tag.
   - **Dense Mode** (e.g., HotpotQA/FEVER): Strict alternation Thought → Action → Observation (fixed steps).
   - **Sparse Mode** (e.g., ALFWorld/WebShop): Model decides when to think or act asynchronously.
   - **Synergy**:
     - Reason → Act: Thoughts guide actions (planning, reformulating queries).
     - Act → Reason: Observations ground reasoning (reduce hallucinations).
   - Loop until `Finish[answer]` or max steps.

3. **Verbalizer**: Maps actions to environment interfaces (discrete actions):
   | Task       | Example Actions                  |
   |------------|----------------------------------|
   | HotpotQA  | `search[entity]`, `lookup[string]`, `finish[answer]` |
   | FEVER     | Same as HotpotQA                |
   | ALFWorld  | `go[location]`, `open[object]`  |
   | WebShop   | `click[item]`, `type[key]`      |

4. **Integration**: Black-box environments (e.g., Wikipedia API simulator, text game sims). Observations are 
textualized and fed back. Supports human-in-the-loop editing.

**Advantages**: Interpretable trajectories, few-shot performance beating imitation learning (IL) baselines trained 
on 10^3-10^5 samples, general across QA and embodied tasks.

### Metrics Reported by Authors (PaLM-540B unless noted)

#### HotpotQA (Multi-hop QA, 6-shot, EM %):
| Method          | EM   |
|-----------------|------|
| Standard        | 28.7 |
| CoT             | 29.4 |
| Act-only        | 25.7 |
| **ReAct**       | **27.4** |
| ReAct + CoT-SC  | **35.1** (best) |

GPT-3 (davinci-002): 30.4 EM.

#### FEVER (Fact Verification, 3-shot, Acc %):
| Method          | Acc  |
|-----------------|------|
| Standard        | 57.1 |
| CoT             | 56.3 |
| Act-only        | 58.9 |
| **ReAct**       | **60.9** |
| CoT-SC + ReAct  | **64.6** (best) |

GPT-3: 54%.

#### ALFWorld (Text-based Games, Success %, 2-shot):
| Method     | Success |
|------------|---------|
| Act-only   | 45%    |
| **ReAct**  | **71%** (70.9%) |
| IL Baselines | 37% |

GPT-3: 78.4%.

#### WebShop (Web Navigation, Success %, 1-shot):
| Method     | Success |
|------------|---------|
| Act-only   | 30.1%  |
| **ReAct**  | **40%** |
| IL Baselines | 29.1% |

GPT-3: 35.8%.

**Human Eval (HotpotQA)**: ReAct had 0% hallucination failures (vs CoT 56%), more grounded reasoning.

Paper: [arXiv:2210.03629](https://arxiv.org/abs/2210.03629), ICLR 2023.

[Step 2: Duration 14.85 seconds| Input tokens: 7,217 | Output tokens: 1,751]

In [65]:
print(answer)


### How the ReAct Agent Works

ReAct (Reasoning + Acting in Language Models) is a prompting framework that synergizes verbal reasoning (thoughts) and task-specific actions in large language models (LLMs like PaLM-540B or GPT-3). It interleaves **Thought** (free-form reasoning traces) and **Action** (e.g., `search[entity]`) with **Observation** from external tools/world models, enabling dynamic interaction to solve complex tasks like multi-hop QA, fact verification, and decision-making in simulated environments.

#### Key Components:
1. **Prompt Structure**: Few-shot in-context learning (1-6 examples per task). Each example shows a full trajectory:
   ```
   Question: ...?
   Thought: I need to find X by searching Y first.
   Action: search[Y]
   Observation: [Environment response, e.g., Wikipedia snippet]
   Thought: From the observation, Z is related...
   Action: lookup[key_term]
   Observation: ...
   Thought: The answer is A.
   Finish[A]
   ```
   Thoughts are natural language fo

In [66]:
from IPython.display import display, Markdown, Latex

display(Markdown(answer))


### How the ReAct Agent Works

ReAct (Reasoning + Acting in Language Models) is a prompting framework that synergizes verbal reasoning (thoughts) and task-specific actions in large language models (LLMs like PaLM-540B or GPT-3). It interleaves **Thought** (free-form reasoning traces) and **Action** (e.g., `search[entity]`) with **Observation** from external tools/world models, enabling dynamic interaction to solve complex tasks like multi-hop QA, fact verification, and decision-making in simulated environments.

#### Key Components:
1. **Prompt Structure**: Few-shot in-context learning (1-6 examples per task). Each example shows a full trajectory:
   ```
   Question: ...?
   Thought: I need to find X by searching Y first.
   Action: search[Y]
   Observation: [Environment response, e.g., Wikipedia snippet]
   Thought: From the observation, Z is related...
   Action: lookup[key_term]
   Observation: ...
   Thought: The answer is A.
   Finish[A]
   ```
   Thoughts are natural language for planning, fact extraction, progress tracking, and error handling. Actions are task-specific (via verbalizer).

2. **Reasoning-Acting Loop**:
   - **Generation**: LLM autoregressively generates until a [Thought] or [Action] tag.
   - **Dense Mode** (e.g., HotpotQA/FEVER): Strict alternation Thought → Action → Observation (fixed steps).
   - **Sparse Mode** (e.g., ALFWorld/WebShop): Model decides when to think or act asynchronously.
   - **Synergy**:
     - Reason → Act: Thoughts guide actions (planning, reformulating queries).
     - Act → Reason: Observations ground reasoning (reduce hallucinations).
   - Loop until `Finish[answer]` or max steps.

3. **Verbalizer**: Maps actions to environment interfaces (discrete actions):
   | Task       | Example Actions                  |
   |------------|----------------------------------|
   | HotpotQA  | `search[entity]`, `lookup[string]`, `finish[answer]` |
   | FEVER     | Same as HotpotQA                |
   | ALFWorld  | `go[location]`, `open[object]`  |
   | WebShop   | `click[item]`, `type[key]`      |

4. **Integration**: Black-box environments (e.g., Wikipedia API simulator, text game sims). Observations are textualized and fed back. Supports human-in-the-loop editing.

**Advantages**: Interpretable trajectories, few-shot performance beating imitation learning (IL) baselines trained on 10^3-10^5 samples, general across QA and embodied tasks.

### Metrics Reported by Authors (PaLM-540B unless noted)

#### HotpotQA (Multi-hop QA, 6-shot, EM %):
| Method          | EM   |
|-----------------|------|
| Standard        | 28.7 |
| CoT             | 29.4 |
| Act-only        | 25.7 |
| **ReAct**       | **27.4** |
| ReAct + CoT-SC  | **35.1** (best) |

GPT-3 (davinci-002): 30.4 EM.

#### FEVER (Fact Verification, 3-shot, Acc %):
| Method          | Acc  |
|-----------------|------|
| Standard        | 57.1 |
| CoT             | 56.3 |
| Act-only        | 58.9 |
| **ReAct**       | **60.9** |
| CoT-SC + ReAct  | **64.6** (best) |

GPT-3: 54%.

#### ALFWorld (Text-based Games, Success %, 2-shot):
| Method     | Success |
|------------|---------|
| Act-only   | 45%    |
| **ReAct**  | **71%** (70.9%) |
| IL Baselines | 37% |

GPT-3: 78.4%.

#### WebShop (Web Navigation, Success %, 1-shot):
| Method     | Success |
|------------|---------|
| Act-only   | 30.1%  |
| **ReAct**  | **40%** |
| IL Baselines | 29.1% |

GPT-3: 35.8%.

**Human Eval (HotpotQA)**: ReAct had 0% hallucination failures (vs CoT 56%), more grounded reasoning.

Paper: [arXiv:2210.03629](https://arxiv.org/abs/2210.03629), ICLR 2023.
